# Setup

In [68]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



Python: 3.10.4 (tags/v3.10.4:9d38120, Mar 23 2022, 23:13:41) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [70]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: NVIDIA RTX A2000 8GB Laptop GPU
Using DEVICE = cuda:0


# Imports

In [71]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc


from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [72]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [73]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import require_ocean_layer


In [74]:
from ontology_graph_builder import build_networkx_graph_from_ttl, ontology_label_tree

from cluster_typing.ontology_probability_scoring import (
    OntologyTraversalConfig,
    OntologyScoringConfig,
    OntologyMentionWeightConfig,
    OntologyEvidenceExportConfig,
    export_ontology_evidence_jsonls,
)

from cluster_typing.ontology_annotator import (
    OntologyAnnotationConfig,
    annotate_doc_with_ontology_folder,
)

from cluster_typing.ontology_schema import require_ontology_layer


# Functions

In [75]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [76]:
def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [77]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str = "oz_full.txt",
    local_path: str | Path = "../../datasets/libri/oz_full.txt",
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [78]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [79]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"../../datasets/libri/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
ANNOTATED_DOC_PATH = OUTPUT_ROOT / "annotated_doc.pkl"
ONTOLOGY_TYPED_DOC_PATH = OUTPUT_ROOT / "ontology_typed_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT/ "narrative_entity_ontology_clean.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("ANNOTATED_DOC_PATH =", ANNOTATED_DOC_PATH)
print("ONTOLOGY_TYPED_DOC_PATH =", ONTOLOGY_TYPED_DOC_PATH)
print("OCEAN_SCORED_DOC_PATH =", OCEAN_SCORED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


TEXT_PATH = ..\..\datasets\libri\oz_full.txt
OUTPUT_ROOT = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs
TOKENIZED_DOC_PATH = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\tokenized_doc.pkl
ANNOTATED_DOC_PATH = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\annotated_doc.pkl
ONTOLOGY_TYPED_DOC_PATH = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\ontology_typed_doc.pkl
OCEAN_SCORED_DOC_PATH = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\ocean_scored_doc.pkl
ONTOLOGY_TTL_PATH = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\ontologies\narrative_entity_ontology_clean.ttl


In [80]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\global_coref


## Runtime and pipeline config

In [81]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA RTX A2000 8GB Laptop GPU', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\global_coref


# Main

### Tokenization

In [82]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\tokenized_doc.pkl
Doc tokens: 48045
Doc sentences: 1943


### Chunking

In [83]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Coreference clusters extraction

In [84]:
if ANNOTATED_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {ANNOTATED_DOC_PATH}")
    doc = load_doc(ANNOTATED_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, ANNOTATED_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", ANNOTATED_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

[coref-annotation] Loading coref-annotated Doc from c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\annotated_doc.pkl


In [85]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

[coref-annotation] Summary:
{'n_mentions': 9460, 'n_clusters': 566, 'n_indexed_tokens': 11462, 'n_indexed_heads': 7346, 'n_indexed_spans': 7375}
Non-singleton clusters: 199

cluster_id=8 | canonical_name='Dorothy' | n_mentions=1900

cluster_id=97 | canonical_name='the Scarecrow' | n_mentions=1032

cluster_id=153 | canonical_name='the Lion' | n_mentions=837

cluster_id=48 | canonical_name='Oz' | n_mentions=801

cluster_id=111 | canonical_name='the travelers' | n_mentions=598

cluster_id=76 | canonical_name='the Wicked Witch' | n_mentions=357

cluster_id=226 | canonical_name='the Tin Woodman' | n_mentions=258

cluster_id=20 | canonical_name='Toto' | n_mentions=179

cluster_id=60 | canonical_name='the Emerald City' | n_mentions=171

cluster_id=41 | canonical_name='her friends' | n_mentions=140

cluster_id=4 | canonical_name='Aunt Em' | n_mentions=117

cluster_id=252 | canonical_name='the people' | n_mentions=99

cluster_id=67 | canonical_name='the Winkies' | n_mentions=94

cluster_id=1 | 

### Ontology cluster typing

In [86]:
# Cluster IDs to ontology-type.
# Keep this independent from OCEAN so the two sub-pipelines can be run separately.
ontology_cluster_ids = [8, 97, 153, 48, 111, 76, 226, 20, 41, 4, 252, 67, 35, 540, 238, 335, 292, 201, 460, 533, 326, 36, 2, 22, 221, 387]
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67


ontology_graph = build_networkx_graph_from_ttl(ONTOLOGY_TTL_PATH)
print("[ontology typing] Loaded ontology graph")
print("nodes:", ontology_graph.number_of_nodes())
print("edges:", ontology_graph.number_of_edges())
print(ontology_label_tree(ontology_graph))

[ontology typing] Loaded ontology graph
nodes: 23
edges: 22
- Narrative Entity
  - Character
    - Human Character
    - Non Human Character
  - Concept
    - Belief Or Value
    - Power Or Ability
    - Rule Or Law
  - Event
    - Action
    - Occurrence
  - Group
    - Informal Group
    - Organization
    - People Or Species
  - Object
    - Document
    - Item
    - Substance
  - Place
    - Region
    - Settlement
    - Site


In [87]:
ontology_n_mentions = None  # None = type ALL mentions for each cluster
ontology_random_seed = 67


if ONTOLOGY_TYPED_DOC_PATH.exists():
    print(f"[ontology typing] Loading ontology-typed Doc from {ONTOLOGY_TYPED_DOC_PATH}")
    doc = load_doc(ONTOLOGY_TYPED_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_ontology_evidence_jsonls(
        doc,
        ontology_graph,
        OntologyEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=OntologyTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=OntologyScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=OntologyMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=False,
            resume_from_jsonl=True,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "ontology_typing" / n_part

    doc = annotate_doc_with_ontology_folder(
        doc,
        ontology_graph,
        ontology_folder,
        config=OntologyAnnotationConfig(
            use_mention_weight=True,
        ),
    )
    save_doc(doc, ONTOLOGY_TYPED_DOC_PATH)
    print("\n\n\n[ontology typing] Saved annotated Doc to:", ONTOLOGY_TYPED_DOC_PATH)

ontology_layer = require_ontology_layer(doc)
print(ontology_layer.summary())

All-cluster ontology evidence export
cluster source: doc._.coref_layer.clusters
n_clusters: 566
cluster_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 5271.70it/s]


[chunk 1/1] mentions=0:1 | chunk_time=29.32s | done_total=1/1
ONTOLOGY EVIDENCE JSONL EXPORT COMPLETE
new records written: 1
skipped from existing JSONL: 0
elapsed scoring/export time: 29.47s
jsonl saved to: c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\ontology_typing\all\ontology_evidence_cluster_0_the_great_Kansas_prairies_all.jsonl

----------------------------------------------------------------------------------------------------
cluster 2/566
cluster_id: 1
subject: 'Kansas'
jsonl_path: c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\ontology_typing\all\ontology_evidence_cluster_1_Kansas_all.jsonl
per_cluster_seed: 8825220446749074408
----------------------------------------------------------------------------------------------------
Ontology evidence JSONL export
cluster_id: 1
subject: 'Kansas'
requested mentions: None
total mentions in cluster: 93
jsonl_path: c:\Users\Bobby\Documents\GitHub\NLP_character_graph_pipeline\outputs\ontolog

KeyboardInterrupt: 

In [ ]:
coref = doc._.coref_layer
ontology_layer = doc._.ontology_layer

for cluster_id in ontology_cluster_ids:
    annotation = ontology_layer.cluster(cluster_id)
    print(
        f"{cluster_id} | "
        f"{coref.clusters[cluster_id].canonical_name!r} -> "
        f"{annotation.class_label} "
        f"({annotation.class_human_readable_label})"
    )


### OCEAN scoring

In [ ]:
cluster_ids = [8, 97, 153, 48, 111, 76, 226, 20, 41, 4, 252, 67, 35, 540, 238, 335, 292, 201, 460, 533, 326, 36, 2, 22, 221, 387]
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [ ]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root="./outputs",

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = Path("./outputs") / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

In [ ]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name)
    print(ocean.cluster(cluster_id).scores.as_dict())